# 1A pilot v3 — preferred answer formats, ChatML for VinaLLaMA-2.7b, ViMMRC NLL on question + options

> **Tóm tắt (VI):** v3 sửa 3 điểm mà bản kiểm tra v2.2 phát hiện, và giữ nguyên mọi thứ khác (cùng câu, cùng nhiễu, cùng cách hiệu chỉnh).
> 1. **Qwen3 và bloomz** đặt rất ít xác suất lên nhãn (Qwen3 trả lời kiểu markdown `**nhãn**`, bloomz trả lời rỗng hoặc tiếng Anh). v3 thử 4 định dạng trả lời trên 40 câu sạch và **chọn định dạng có label mass cao nhất** cho từng (mô hình, tác vụ). Việc chọn **không dùng nhãn đúng**, nên không có rò rỉ.
> 2. **VinaLLaMA-2.7b** không có chat template nên v2.1 cho nó prompt thường, còn bản 7b dùng ChatML. v3 cho bản 2.7b **dùng chung template ChatML của bản 7b**.
> 3. **ViMMRC:** NLL v2.1 tính trên cả bài đọc, câu hỏi và đáp án; bài đọc chiếm phần lớn. v3 tính thêm NLL **chỉ trên câu hỏi + đáp án** cho cả 16 mô hình.
>
> Kết quả được ghi theo đúng cấu trúc thư mục của v2.1, nên notebook phân tích `1A_pilot_v3_analysis.ipynb` (CPU) đọc thẳng được.

**What changes from v2.1**

| v2.1 / v2.2 finding | v3 |
|---|---|
| Qwen3 (all sizes) and bloomz put < 50% of their probability on the label words in 19 (model, task) pairs; their free answers are markdown (`**Tích cực**`), empty, or English (`negative`, `Yes`) | **Answer format chosen per (model, task), label-blind** (below). Only these 6 models are re-scored |
| `vinallama-2.7b-chat` ships without a chat template, so it got a plain prompt while `vinallama-7b-chat` got ChatML: the size comparison mixed size with prompt format | 2.7b gets the **7b's ChatML template**, v2.1 answer format otherwise |
| ViMMRC NLL was over article + question + options; the article is ~83% of the syllables (median), so ΔNLL mostly measured the article | **New columns `nll_qo`, `nll_qo_tokens`, `qo_syl`**: NLL of question + options only, after a short clean context, for **all 16 models** (the 9 other models load only for this) |

**Candidate answer formats** (scored on 40 clean items per task; the one with the highest median label mass wins, `v21` is kept unless another beats it by ≥ 0.02):

| format | prompt | label scored as |
|---|---|---|
| `v21` | instruction, then the input; answer prefilled with "Nhãn:" / "Đáp án:" | `" tiêu cực"`, `" Tiêu cực"` (exactly v2.1) |
| `start` | same prompt, no prefill | the answer's first words: `tiêu cực`, `Tiêu cực`, and for chat models `**Tiêu cực` |
| `q_last` | the input first, the instruction last (bloomz's xP3 training format) | as `start`; models without a chat template may put a space or newline first |
| `q_last_en` | as `q_last` | also the English label words: `negative`, `joy`, `hate`, `yes`/`maybe`/`no`, … |

Every variant of a label adds to that label's probability, exactly as the two casings did in v2.1. Selection looks only at label mass on clean items, never at accuracy or at the gold labels.

**Runtime:** L4 GPU, about **2–3 hours (6–9 CU)**: 7 models re-scored on all 5 tasks, 9 models load only for 1,120 short ViMMRC sequences. Everything is saved per (model, task): if the session stops, Run all again. The gated Llama-3.2-3B needs the `HF_TOKEN` secret, as in v2.1; without it, its ViMMRC `nll_qo` is left empty and the analysis notebook says so.

⚠️ Keep the tab open (Colab disconnects after ~90 min without interaction). When done: `Runtime → Disconnect and delete runtime`.

## 0. Setup

In [ ]:
import sys, os, json
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install -U transformers accelerate

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/llm-research"
else:
    ROOT = os.environ.get("PILOT_ROOT", "outputs")
V21_DIR = f"{ROOT}/1A-pilot-v2.1"   # v2.1 results: everything not re-scored is taken from here
OUT_DIR = f"{ROOT}/1A-pilot-v3"
os.makedirs(f"{OUT_DIR}/preds", exist_ok=True)
assert os.path.exists(f"{V21_DIR}/noisy_items.csv"), f"v2.1 results not found in {V21_DIR}"
print("reading", V21_DIR, "→ writing", OUT_DIR)
# local test runs only: {"model name": "local folder"}; empty on Colab
LOCAL_MODELS = json.loads(os.environ.get("PILOT_LOCAL_MODELS", "{}"))

In [ ]:
# Only needed for gated models. In Colab: 🔑 Secrets panel → HF_TOKEN
HF_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN secret — gated models will be skipped.")

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(p.name, f"{p.total_memory / 1e9:.1f} GB")
    DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
else:
    print("⚠️ No GPU. Runtime → Change runtime type → L4 GPU")
    DTYPE = torch.float32
print("dtype:", DTYPE)

## 1. Config

Same models, tasks, conditions and seed as v2.1. `RESCORE` gets the format search; `CHAT_TEMPLATE_FROM` borrows a chat template; every other model only adds the ViMMRC question + options NLL.

In [ ]:
MODEL_INFO = {  # name: (family, size in billions), in v2.1's order (small to large)
    "Qwen/Qwen2.5-1.5B-Instruct": ("Qwen2.5", 1.5),
    "Qwen/Qwen3-1.7B": ("Qwen3", 1.7),
    "SeaLLMs/SeaLLMs-v3-1.5B-Chat": ("SeaLLMs-v3", 1.5),
    "bigscience/bloomz-1b7": ("bloomz", 1.7),
    "vilm/vinallama-2.7b-chat": ("VinaLLaMA", 2.7),
    "Qwen/Qwen2.5-3B-Instruct": ("Qwen2.5", 3.1),
    "bigscience/bloomz-3b": ("bloomz", 3.0),
    "Qwen/Qwen3-4B": ("Qwen3", 4.0),
    "microsoft/Phi-3.5-mini-instruct": ("Phi-3.5", 3.8),
    "Qwen/Qwen2.5-7B-Instruct": ("Qwen2.5", 7.6),
    "Qwen/Qwen3-8B": ("Qwen3", 8.2),
    "SeaLLMs/SeaLLMs-v3-7B-Chat": ("SeaLLMs-v3", 7.6),
    "vilm/vinallama-7b-chat": ("VinaLLaMA", 6.9),
    "bigscience/bloomz-7b1": ("bloomz", 7.1),
    "mistralai/Mistral-7B-Instruct-v0.3": ("Mistral", 7.2),
    "meta-llama/Llama-3.2-3B-Instruct": ("Llama-3", 3.2),   # gated
}
GATED = {"meta-llama/Llama-3.2-3B-Instruct"}
RESCORE = {"Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B", "Qwen/Qwen3-8B",
           "bigscience/bloomz-1b7", "bigscience/bloomz-3b", "bigscience/bloomz-7b1"}
CHAT_TEMPLATE_FROM = {"vilm/vinallama-2.7b-chat": "vilm/vinallama-7b-chat"}

TASKS = ["vsfc", "vsmec", "vihsd", "xnli", "vimmrc"]
CONDITIONS = ["clean", "partial25", "partial50", "partial75", "strip", "teencode", "mixed"]
SEED = 42
PROBE_N = 40           # clean items per task used to pick the answer format
MIN_GAIN = 0.02        # another format must beat v21's median label mass by this much
TOKEN_BUDGET = 4096    # max padded tokens per NLL batch; halved on out-of-memory
KV_BUDGET = 8192       # max cached prompt tokens when scoring several label continuations at once
DELETE_WEIGHTS_AFTER_USE = IN_COLAB

def v21_path(name, task):
    return f"{V21_DIR}/preds/{task}/{name.replace('/', '__')}.csv"

def pred_path(name, task):
    return f"{OUT_DIR}/preds/{task}/{name.replace('/', '__')}.csv"

MODELS = [m for m in MODEL_INFO if all(os.path.exists(v21_path(m, t)) for t in TASKS)]
print("models in the v2.1 run:", len(MODELS))
print("missing from v2.1 (skipped):", [m for m in MODEL_INFO if m not in MODELS] or "none")

## 2. Items: rebuilt with v2.1's code, then checked against v2.1's file

The noise generator and task definitions below are copied unchanged from v2.1. The rebuilt items must equal `noisy_items.csv` from the v2.1 run, otherwise the notebook stops.

In [ ]:
import random, re, unicodedata, ast, shutil
import numpy as np
import pandas as pd

def strip_diacritics(s):
    s = s.replace("đ", "d").replace("Đ", "D")
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return unicodedata.normalize("NFC", s)

def partial_strip(text, rate, rng):
    words = text.split(" ")
    candidates = [i for i, w in enumerate(words) if strip_diacritics(w) != w]
    chosen = set(rng.sample(candidates, round(rate * len(candidates))))
    return " ".join(strip_diacritics(w) if i in chosen else w for i, w in enumerate(words))

TEENCODE = {
    "giáo viên": "gv", "giảng viên": "gv", "sinh viên": "sv", "bài tập": "bt", "học sinh": "hs",
    "không": "ko", "được": "dc", "gì": "j", "với": "vs", "mình": "mk", "rồi": "r",
    "vậy": "v", "bạn": "bn", "quá": "wa", "nhiều": "nhìu", "người": "ng", "trước": "trc",
    "cũng": "cx", "thôi": "thui", "biết": "bít", "yêu": "iu", "giờ": "h",
    "nhưng": "nhg", "lắm": "lém", "thích": "thik", "những": "nhữg", "hiểu": "hỉu",
}
_teen_re = re.compile(
    r"(?<!\w)(" + "|".join(sorted(map(re.escape, TEENCODE), key=len, reverse=True)) + r")(?!\w)",
    re.IGNORECASE,
)

def teencode(text):
    return _teen_re.sub(lambda m: TEENCODE[m.group(0).lower()], text)

def make_noisy(text, cond, item_id, field_idx=0):
    rng = random.Random(f"{SEED}-{item_id}-{cond}" + (f"-{field_idx}" if field_idx else ""))
    if cond == "clean":
        return text
    if cond == "strip":
        return strip_diacritics(text)
    if cond.startswith("partial"):
        return partial_strip(text, int(cond[len("partial"):]) / 100, rng)
    if cond == "teencode":
        return teencode(text)
    if cond == "mixed":
        return partial_strip(teencode(text), 0.5, rng)
    raise ValueError(cond)

def syllable_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text) if any(ch.isalpha() for ch in m.group())]

PARQUET = "https://huggingface.co/datasets/{}/resolve/refs%2Fconvert%2Fparquet/{}/test/0000.parquet"

TASK_SPECS = {
    "vsfc": dict(
        source=("uitnlp/vietnamese_students_feedback", "default"), fields=["sentence"], label_col="sentiment",
        classes=[0, 1, 2], labels=["tiêu cực", "trung tính", "tích cực"], per_class=100,
        instruction=("Phân loại cảm xúc của câu nhận xét sau của sinh viên. "
                     "Trả lời bằng đúng một nhãn: tiêu cực, trung tính, hoặc tích cực.\n\nCâu: {sentence}"),
        prefill="Nhãn:", nll_context="Nhận xét của sinh viên:"),
    "vsmec": dict(
        source=("ura-hcmut/UIT-VSMEC", "default"), fields=["Sentence"], label_col="Emotion",
        classes=["Enjoyment", "Sadness", "Anger", "Fear", "Disgust", "Surprise", "Other"],
        labels=["vui vẻ", "buồn bã", "tức giận", "sợ hãi", "ghê tởm", "ngạc nhiên", "khác"], per_class=35,
        instruction=("Câu bình luận sau thể hiện cảm xúc gì? Trả lời bằng đúng một nhãn: "
                     "vui vẻ, buồn bã, tức giận, sợ hãi, ghê tởm, ngạc nhiên, hoặc khác.\n\nCâu: {Sentence}"),
        prefill="Nhãn:", nll_context="Bình luận trên mạng xã hội:"),
    "vihsd": dict(
        source=("ura-hcmut/UIT-ViHSD", "default"), fields=["free_text"], label_col="label_id",
        classes=[0, 1, 2], labels=["bình thường", "xúc phạm", "thù ghét"], per_class=100,
        instruction=("Phân loại bình luận mạng xã hội sau. Trả lời bằng đúng một nhãn: "
                     "bình thường (không có lời lẽ xấu), xúc phạm (có lời lẽ thô tục nhưng không nhắm vào ai), "
                     "hoặc thù ghét (công kích, lăng mạ một người hay một nhóm người).\n\nBình luận: {free_text}"),
        prefill="Nhãn:", nll_context="Bình luận trên mạng xã hội:"),
    "xnli": dict(
        source=("facebook/xnli", "vi"), fields=["premise", "hypothesis"], label_col="label",
        classes=[0, 1, 2], labels=["kéo theo", "trung lập", "mâu thuẫn"], per_class=100,
        instruction=("Cho tiền đề và giả thuyết sau. Giả thuyết có được suy ra từ tiền đề không? "
                     "Trả lời bằng đúng một nhãn: kéo theo (giả thuyết chắc chắn đúng), trung lập (không xác định được), "
                     "hoặc mâu thuẫn (giả thuyết chắc chắn sai).\n\nTiền đề: {premise}\nGiả thuyết: {hypothesis}"),
        prefill="Nhãn:", nll_context="Văn bản:"),
    "vimmrc": dict(
        source=("ura-hcmut/ViMMRC", "default"), fields=["article", "question", "options"], label_col="answer",
        classes=["A", "B", "C", "D"], labels=["A", "B", "C", "D"], per_class=40,
        instruction=("Đọc đoạn văn và chọn đáp án đúng cho câu hỏi. Trả lời bằng đúng một chữ cái: A, B, C hoặc D.\n\n"
                     "Đoạn văn: {article}\n\nCâu hỏi: {question}\n{options}"),
        prefill="Đáp án:", nll_context="Bài đọc:"),
}

def load_task(task):
    spec = TASK_SPECS[task]
    raw = pd.read_parquet(PARQUET.format(*spec["source"]))
    if task == "vimmrc":
        raw["options"] = [ "\n".join(f"{l}. {o}" for l, o in zip("ABCD", ast.literal_eval(x))) for x in raw.options]
    raw = raw[raw[spec["label_col"]].isin(spec["classes"])]
    sample = raw.groupby(spec["label_col"]).sample(n=spec["per_class"], random_state=SEED).reset_index(drop=True)
    rows = []
    for item_id, r in enumerate(sample.itertuples(index=False)):
        r = r._asdict()
        label = spec["classes"].index(r[spec["label_col"]])
        for c in CONDITIONS:
            row = {"task": task, "item_id": item_id, "label": label, "cond": c}
            for fi, f in enumerate(spec["fields"]):
                row[f] = make_noisy(r[f], c, item_id, fi)
                row[f"{f}_clean"] = r[f]
            row["text"] = "\n".join(row[f] for f in spec["fields"])
            row["clean"] = "\n".join(row[f"{f}_clean"] for f in spec["fields"])
            rows.append(row)
    return pd.DataFrame(rows)

DATA = {t: load_task(t) for t in TASKS}
# ViMMRC: the question + options part, scored on its own for the new NLL
v = DATA["vimmrc"]
v["qo"] = v.question + "\n" + v.options
v["qo_clean"] = v.question_clean + "\n" + v.options_clean

cols = ["task", "item_id", "cond", "text", "clean"]
ran = pd.read_csv(f"{V21_DIR}/noisy_items.csv", keep_default_na=False)[cols]
mine = pd.concat(DATA.values())[cols]
key = ["task", "item_id", "cond"]
same = ran.sort_values(key).reset_index(drop=True).equals(mine.sort_values(key).reset_index(drop=True))
assert same, "rebuilt items differ from the v2.1 run: stop here"
mine.to_csv(f"{OUT_DIR}/noisy_items.csv", index=False)
print(f"{len(mine)} rows identical to the v2.1 run ✓")
qo_share = (v[v.cond == "clean"].qo_clean.map(lambda s: len(syllable_spans(s)))
            / v[v.cond == "clean"].clean.map(lambda s: len(syllable_spans(s)))).median()
print(f"ViMMRC: question + options = {qo_share:.0%} of an item's syllables (median)")

In [ ]:
# Tokenizer statistics only depend on the text, so v2.1's are reused as they are
os.makedirs(f"{OUT_DIR}/tokenizer", exist_ok=True)
for m in MODELS:
    shutil.copy(f"{V21_DIR}/tokenizer/{m.replace('/', '__')}.csv", f"{OUT_DIR}/tokenizer/")
shutil.copy(f"{V21_DIR}/tokenizer_stats.csv", OUT_DIR)
print("tokenizer statistics copied for", len(MODELS), "models")

## 3. Answer formats

A format fixes three things: where the instruction goes (`layout`), what the answer is prefilled with (`prefill`), and which strings count as the start of each label. Separators and markdown markers depend on the model: after a chat template the answer starts right away, so chat models get no separator and an optional `**`; plain prompts (bloomz) get a space or a newline.

Labels are tokenized **in context**, as in v2.1: the tokens of (prompt + variant) minus the tokens of the prompt.

In [ ]:
EN_LABELS = {  # one English word per label, the words bloomz answered with in v2.1
    "vsfc": [["negative"], ["neutral"], ["positive"]],
    "vsmec": [["joy"], ["sadness"], ["anger"], ["fear"], ["disgust"], ["surprise"], ["other"]],
    "vihsd": [["normal"], ["offensive"], ["hate"]],
    "xnli": [["yes"], ["maybe"], ["no"]],
    "vimmrc": [[], [], [], []],
}
FORMATS = {  # prefill None = the task's v2.1 prefill; seps/markers None = chosen by prompt type
    "v21":       dict(layout="instruction_first", prefill=None, seps=[" "], markers=[""], en=False),
    "start":     dict(layout="instruction_first", prefill="", seps=None, markers=None, en=False),
    "q_last":    dict(layout="question_last", prefill="", seps=None, markers=None, en=False),
    "q_last_en": dict(layout="question_last", prefill="", seps=None, markers=None, en=True),
}

def user_message(task, row, layout):
    spec = TASK_SPECS[task]
    instr, body = spec["instruction"].split("\n\n", 1)
    body = body.format(**{f: row[f] for f in spec["fields"]})
    return f"{instr}\n\n{body}" if layout == "instruction_first" else f"{body}\n\n{instr}"

def prompt_text(tok, task, row, fmt, prefill=True):
    f = FORMATS[fmt]
    msg = user_message(task, row, f["layout"])
    pre = (TASK_SPECS[task]["prefill"] if f["prefill"] is None else f["prefill"]) if prefill else ""
    if tok.chat_template:
        s = tok.apply_chat_template([{"role": "user", "content": msg}], tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False)
        return s + pre, False
    return (f"{msg}\n{pre}" if pre else msg), True

def variant_texts(task, fmt, chat):
    # per label, every string that counts as answering with that label
    f = FORMATS[fmt]
    seps = f["seps"] or ([""] if chat else [" ", "\n"])
    markers = f["markers"] or (["", "**"] if chat else [""])
    out = []
    for k, vi in enumerate(TASK_SPECS[task]["labels"]):
        words = [vi] + (EN_LABELS[task][k] if f["en"] else [])
        cased = dict.fromkeys(c for w in words for c in (w, w[0].upper() + w[1:]))
        out.append([s + m + w for s in seps for m in markers for w in cased])
    return out

def in_context_ids(tok, prompt, add_special, p_ids, texts):
    ids, fallbacks = [], 0
    for v in texts:
        full = tok(prompt + v, add_special_tokens=add_special)["input_ids"]
        if full[:len(p_ids)] == p_ids and len(full) > len(p_ids):
            ids.append(tuple(full[len(p_ids):]))
        else:  # the variant merged into the prompt's last token; fall back to the variant alone
            ids.append(tuple(tok(v, add_special_tokens=False)["input_ids"]))
            fallbacks += 1
    return list(dict.fromkeys(ids)), fallbacks

# v21 must be v2.1's prompt, word for word
_r = DATA["xnli"].iloc[0].to_dict()
assert user_message("xnli", _r, "instruction_first") == TASK_SPECS["xnli"]["instruction"].format(**_r)
for fmt in FORMATS:
    print(f"{fmt:10} chat:", variant_texts("vsfc", fmt, True)[0], "| plain:", variant_texts("vsfc", fmt, False)[0])

## 4. Scoring

- **Labels:** one forward pass over the prompt, kept in the KV cache; every label variant is then scored as a continuation of that cache. The first token of every variant comes from the prompt pass itself; only multi-token variants need the second pass. The result is the same as scoring (prompt + variant) from scratch, as in v2.1, at a fraction of the cost.
- **NLL:** unchanged from v2.1 (text after a short clean context, batched by token budget). `nll_qo` uses the context "Câu hỏi trắc nghiệm:" and scores question + options.
- **Calibration:** unchanged (subtract the mean label distribution of the same condition).

In [ ]:
import gc, time, copy
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import constants as hf_constants

def expand_cache(cache, n):
    if hasattr(cache, "batch_repeat_interleave"):
        c = copy.deepcopy(cache)
        c.batch_repeat_interleave(n)
        return c
    return tuple(tuple(x.repeat_interleave(n, 0) for x in layer) for layer in cache)

@torch.no_grad()
def score_continuations(model, p_ids, variants, kv_budget):
    # log p(variant | prompt) for every variant (a tuple of token ids)
    out = model(input_ids=torch.tensor([p_ids], device=model.device), use_cache=True)
    first = out.logits[0, -1].float().log_softmax(-1)
    res = first[torch.tensor([v[0] for v in variants], device=first.device)].clone()
    multi = [i for i, v in enumerate(variants) if len(v) > 1]
    step = max(1, kv_budget // len(p_ids))
    for c in range(0, len(multi), step):
        idx = multi[c:c + step]
        L = max(len(variants[i]) for i in idx) - 1
        feed = torch.zeros((len(idx), L), dtype=torch.long)
        tgt = torch.zeros((len(idx), L), dtype=torch.long)
        m = torch.zeros((len(idx), L), dtype=torch.long)
        for j, i in enumerate(idx):
            v = variants[i]
            feed[j, :len(v) - 1] = torch.tensor(v[:-1])
            tgt[j, :len(v) - 1] = torch.tensor(v[1:])
            m[j, :len(v) - 1] = 1
        feed, tgt, m = feed.to(model.device), tgt.to(model.device), m.to(model.device)
        attn = torch.cat([torch.ones((len(idx), len(p_ids)), dtype=torch.long, device=model.device), m], 1)
        logits = model(input_ids=feed, attention_mask=attn, past_key_values=expand_cache(out.past_key_values, len(idx)),
                       use_cache=False).logits.float()
        lp = (logits.gather(2, tgt[..., None]).squeeze(-1) - logits.logsumexp(-1)) * m
        res[torch.tensor(idx, device=res.device)] += lp.sum(1)
    return res.cpu()

def label_logprobs(model, tok, task, fmt, rows, kv_budget):
    K = len(TASK_SPECS[task]["labels"])
    texts = variant_texts(task, fmt, bool(tok.chat_template))
    lp, fallbacks, n_var = torch.empty((len(rows), K)), 0, 0
    for i, r in enumerate(tqdm(rows, desc=f"{task}/{fmt}", leave=False, mininterval=10)):
        s, add = prompt_text(tok, task, r, fmt)
        p = tok(s, add_special_tokens=add)["input_ids"]
        flat, owner = [], []
        for k, ts in enumerate(texts):
            ids, fb = in_context_ids(tok, s, add, p, ts)
            fallbacks += fb
            flat += ids
            owner += [k] * len(ids)
        sc = score_continuations(model, p, flat, kv_budget)
        owner = torch.tensor(owner)
        lp[i] = torch.stack([sc[owner == k].logsumexp(0) for k in range(K)])
        n_var = max(n_var, len(flat))
    return lp, fallbacks, n_var

def nll_ids(tok, ctx, text):
    enc = tok(f"{ctx} {text}", add_special_tokens=True, return_offsets_mapping=True)
    first = next(i for i, (a, b) in enumerate(enc["offset_mapping"]) if b > a and a >= len(ctx))
    return enc["input_ids"], first

def pad_batch(seqs, pad_id, device):
    L = max(map(len, seqs))
    ids = torch.full((len(seqs), L), pad_id)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        mask[i, :len(s)] = 1
    return ids.to(device), mask.to(device)

@torch.no_grad()
def score(model, pad_id, seqs, starts):
    ids, mask = pad_batch(seqs, pad_id, model.device)
    logits = model(input_ids=ids, attention_mask=mask).logits
    pick = torch.zeros_like(mask, dtype=torch.bool)
    for i, (s, st) in enumerate(zip(seqs, starts)):
        pick[i, st - 1:len(s) - 1] = True
    sel = logits[pick]
    del logits
    tgt = ids[:, 1:][pick[:, :-1]]
    tok_lp = torch.cat([
        sel[i:i + 1024].gather(1, tgt[i:i + 1024, None]).squeeze(1).float() - sel[i:i + 1024].float().logsumexp(-1)
        for i in range(0, len(sel), 1024)])
    owner = pick.nonzero()[:, 0]
    sums = torch.zeros(len(seqs), device=tok_lp.device).index_add_(0, owner, tok_lp)
    return sums.cpu().tolist(), pick.sum(1).cpu().tolist()

def token_batches(items, length, budget):
    order = sorted(range(len(items)), key=lambda i: -length(items[i]))
    batch, longest = [], 0
    for i in order:
        n = length(items[i])
        if batch and max(longest, n) * (len(batch) + 1) > budget:
            yield [items[j] for j in batch]
            batch, longest = [], 0
        batch.append(i)
        longest = max(longest, n)
    if batch:
        yield [items[j] for j in batch]

QO_CONTEXT = "Câu hỏi trắc nghiệm:"

def qo_nll(model, tok, budget):
    # NLL of ViMMRC question + options only, per row of DATA["vimmrc"]
    data = DATA["vimmrc"]
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else (tok.eos_token_id or 0)
    items = [nll_ids(tok, QO_CONTEXT, q) + (i,) for i, q in enumerate(data.qo)]
    nll, ntok = [0.0] * len(items), [0] * len(items)
    for chunk in token_batches(items, lambda x: len(x[0]), budget):
        sums, counts = score(model, pad_id, [x[0] for x in chunk], [x[1] for x in chunk])
        for (_, _, i), s, c in zip(chunk, sums, counts):
            nll[i], ntok[i] = -s, c
    return data[["item_id", "cond"]].assign(nll_qo=nll, nll_qo_tokens=ntok,
                                            qo_syl=[len(syllable_spans(c)) for c in data.qo_clean])

@torch.no_grad()
def greedy_answers(model, tok, task, rows, fmt, max_new_tokens=20):
    # free answers without prefill, as in v2.1, but with the chosen layout
    out = []
    for r in rows:
        s, add = prompt_text(tok, task, r, fmt, prefill=False)
        ids = torch.tensor([tok(s, add_special_tokens=add)["input_ids"]], device=model.device)
        gen = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
        out.append(tok.decode(gen[0, ids.shape[1]:], skip_special_tokens=True).strip())
    return out

In [ ]:
def probe_rows(task):
    clean = DATA[task][DATA[task].cond == "clean"]
    ids = np.sort(clean.item_id.unique())
    pick = ids[np.linspace(0, len(ids) - 1, min(PROBE_N, len(ids))).round().astype(int)]
    return clean[clean.item_id.isin(pick)].to_dict("records")

def probe_formats(model, tok, name, task, kv_budget):
    # label-blind: only the label mass on clean items is used, never the gold label
    rows = probe_rows(task)
    old = pd.read_csv(v21_path(name, task)).set_index(["item_id", "cond"])
    old_mass = old.loc[[(r["item_id"], "clean") for r in rows], "label_mass"].to_numpy()
    out = []
    for fmt in FORMATS:
        lp, fb, n_var = label_logprobs(model, tok, task, fmt, rows, kv_budget)
        mass = lp.logsumexp(-1).exp().numpy()
        row = {"model": name, "task": task, "format": fmt, "median_mass": np.median(mass), "mean_mass": mass.mean(),
               "variants_per_item": n_var, "fallbacks": fb, "items": len(rows)}
        if fmt == "v21":  # must reproduce v2.1's stored label mass (small differences: batching, bf16)
            row["v21_repro_max_abs_dlog"] = np.abs(np.log(mass) - np.log(old_mass)).max()
        out.append(row)
    return pd.DataFrame(out)

def choose_format(probe):
    p = probe.set_index("format").median_mass
    best = p.idxmax()
    return best if p[best] >= p["v21"] + MIN_GAIN else "v21"

def calibrate(lp, data):
    logp = lp.log_softmax(-1)
    cal = torch.empty_like(logp)
    for c in CONDITIONS:
        idx = torch.tensor((data.cond == c).values)
        cal[idx] = logp[idx] - logp[idx].exp().mean(0).log()
    return cal.argmax(1)

def evaluate_task(model, tok, name, task, fmt, kv_budget):
    spec, data = TASK_SPECS[task], DATA[task]
    K = len(spec["labels"])
    rows = data.to_dict("records")
    lp, fallbacks, _ = label_logprobs(model, tok, task, fmt, rows, kv_budget)
    out = data[["task", "item_id", "label", "cond"]].copy()
    out["model"] = name
    for k in range(K):
        out[f"lp{k}"] = lp[:, k].tolist()
    out["label_mass"] = lp.logsumexp(-1).exp().tolist()
    out["pred_raw"] = lp.argmax(1).tolist()
    out["pred"] = calibrate(lp, data).tolist()
    # text NLL does not depend on the prompt: taken from v2.1
    old = pd.read_csv(v21_path(name, task))[["item_id", "cond", "nll", "nll_tokens", "clean_syl"]]
    out = out.merge(old, on=["item_id", "cond"], how="left", validate="one_to_one")
    probe = [r for r in rows if r["cond"] == "clean"]
    probe = [next(r for r in probe if r["label"] == k) for k in range(K)][:6]
    diag = pd.DataFrame({"model": name, "task": task, "item_id": [r["item_id"] for r in probe],
                         "label": [spec["labels"][r["label"]] for r in probe],
                         "text": [r["clean"][:150] for r in probe],
                         "answer": greedy_answers(model, tok, task, probe, fmt), "format": fmt})
    return out, diag, fallbacks

## 5. Run

For each model, in v2.1's order:
- **re-scored models** (Qwen3, bloomz): format probe per task → full run with the chosen format;
- **VinaLLaMA-2.7b**: the 7b's ChatML template, v2.1 format;
- **all other models**: v2.1 predictions copied; the model is loaded only to add `nll_qo` to ViMMRC.

Finished (model, task) files are skipped, so Run all continues where it stopped.

In [ ]:
for t in TASKS:
    os.makedirs(f"{OUT_DIR}/preds/{t}", exist_ok=True)
PROBE_PATH, NEW_DIAG = f"{OUT_DIR}/format_probe.csv", f"{OUT_DIR}/diagnostics_v3_new.csv"
FORMAT_PATH = f"{OUT_DIR}/formats.csv"

def append_csv(df, path):
    df.to_csv(path, index=False, mode="a", header=not os.path.exists(path))

def has_qo(name):
    p = pred_path(name, "vimmrc")
    return os.path.exists(p) and "nll_qo" in pd.read_csv(p, nrows=1).columns

def load(name):
    src = LOCAL_MODELS.get(name, name)
    tok = AutoTokenizer.from_pretrained(src, token=HF_TOKEN)
    if name in CHAT_TEMPLATE_FROM:
        donor = CHAT_TEMPLATE_FROM[name]
        tok.chat_template = AutoTokenizer.from_pretrained(LOCAL_MODELS.get(donor, donor), token=HF_TOKEN).chat_template
        for s in ["<|im_start|>", "<|im_end|>"]:
            ids = tok(s, add_special_tokens=False)["input_ids"]
            print(f"   {s} → {ids} {'(one token ✓)' if len(ids) == 1 else '(split into pieces: the model never saw it as one token)'}")
    model = AutoModelForCausalLM.from_pretrained(src, dtype=DTYPE, token=HF_TOKEN).to(DEVICE).eval()
    return tok, model

def with_oom_retry(fn, label):
    budgets = [KV_BUDGET, TOKEN_BUDGET]
    while True:
        try:
            return fn(*budgets)
        except torch.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            if min(budgets) < 1000:
                raise
            budgets = [b // 2 for b in budgets]
            print(f"   out of memory on {label}, retrying with budgets {budgets}")

failed = {}
for name in MODELS:
    mode = "rescore" if name in RESCORE else ("chat_template" if name in CHAT_TEMPLATE_FROM else "nll_qo")
    if mode == "nll_qo":  # predictions unchanged: copy them, only ViMMRC gets the new NLL
        for t in TASKS:
            if t != "vimmrc" and not os.path.exists(pred_path(name, t)):
                shutil.copy(v21_path(name, t), pred_path(name, t))
        todo = [] if has_qo(name) else ["vimmrc"]
    else:
        todo = [t for t in TASKS if not (os.path.exists(pred_path(name, t)) and (t != "vimmrc" or has_qo(name)))]
    if not todo:
        print("✓ already done:", name)
        continue
    if name in GATED and not HF_TOKEN and name not in LOCAL_MODELS:
        shutil.copy(v21_path(name, "vimmrc"), pred_path(name, "vimmrc"))
        print(f"✗ {name}: gated and no HF_TOKEN, v2.1 files copied without nll_qo")
        continue
    t0 = time.time()
    try:
        tok, model = load(name)
    except Exception as e:
        failed[name] = repr(e)[:300]
        print("✗ failed to load:", name, failed[name])
        continue
    print(f"● {name} [{mode}] loaded ({time.time() - t0:.0f}s)")
    for t in todo:
        t1 = time.time()
        try:
            if mode == "nll_qo":
                qo = with_oom_retry(lambda kv, tb: qo_nll(model, tok, tb), t)
                pd.read_csv(v21_path(name, t)).merge(qo, on=["item_id", "cond"], how="left", validate="one_to_one").to_csv(pred_path(name, t), index=False)
                print(f"   ✓ vimmrc nll_qo ({time.time() - t1:.0f}s)")
                continue
            if mode == "rescore":
                done = pd.read_csv(PROBE_PATH) if os.path.exists(PROBE_PATH) else pd.DataFrame(columns=["model", "task"])
                probe = done[(done.model == name) & (done.task == t)]
                if probe.empty:
                    probe = with_oom_retry(lambda kv, tb: probe_formats(model, tok, name, t, kv), t)
                    append_csv(probe, PROBE_PATH)
                fmt = choose_format(probe)
                print(f"   {t}: median label mass by format", probe.set_index("format").median_mass.round(3).to_dict(), "→", fmt)
            else:
                fmt = "v21"
            res, diag, fb = with_oom_retry(lambda kv, tb: evaluate_task(model, tok, name, t, fmt, kv), t)
            if t == "vimmrc":
                res = res.merge(with_oom_retry(lambda kv, tb: qo_nll(model, tok, tb), t), on=["item_id", "cond"], how="left")
        except Exception as e:
            failed[f"{name} / {t}"] = repr(e)[:300]
            print(f"   ✗ {t} failed:", failed[f"{name} / {t}"])
            continue
        res.to_csv(pred_path(name, t), index=False)
        append_csv(diag, NEW_DIAG)
        append_csv(pd.DataFrame([{"model": name, "task": t, "format": fmt, "fallbacks": fb}]), FORMAT_PATH)
        acc = (res.pred == res.label).groupby(res.cond).mean()
        print(f"   ✓ {t:7} ({time.time() - t1:.0f}s)  format={fmt}  clean={acc['clean']:.2f}  strip={acc['strip']:.2f}  "
              f"label_mass={res[res.cond == 'clean'].label_mass.median():.1%}  chance={1 / len(TASK_SPECS[t]['labels']):.2f}"
              + (f"  ({fb} label variants fell back to tokenizing alone)" if fb else ""))
        for r in diag.itertuples():
            print(f"       [{r.label}] → {r.answer[:60]!r}")
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    if DELETE_WEIGHTS_AFTER_USE:
        shutil.rmtree(os.path.join(hf_constants.HF_HUB_CACHE, "models--" + name.replace("/", "--")), ignore_errors=True)
if failed:
    print("\nFailed:", failed)

In [ ]:
# diagnostics.csv in v2.1's layout: new free answers for the re-scored models, v2.1's for the rest
new = pd.read_csv(NEW_DIAG, keep_default_na=False) if os.path.exists(NEW_DIAG) else pd.DataFrame(columns=["model"])
new = new.drop_duplicates(["model", "task", "item_id"], keep="last")
old = pd.read_csv(f"{V21_DIR}/diagnostics.csv", keep_default_na=False)
old = old[~old.model.isin(new.model.unique())].assign(format="v21")
pd.concat([old, new]).to_csv(f"{OUT_DIR}/diagnostics.csv", index=False)
status = pd.DataFrame([{"model": m, "task": t, "file": os.path.exists(pred_path(m, t))} for m in MODELS for t in TASKS])
print("prediction files:", status.file.sum(), "/", len(status), "| ViMMRC files with nll_qo:", sum(has_qo(m) for m in MODELS), "/", len(MODELS))

## 6. What changed

### Format choice

Median label mass on the 40 clean probe items, per candidate format. `v21_repro` is the largest |Δ log label mass| between the `v21` candidate here and the stored v2.1 file for the same items: it should be small (bf16 and batching noise), which shows the new scorer reproduces v2.1.

In [ ]:
SHORT = {m: m.split("/")[1] for m in MODEL_INFO}
probe = pd.read_csv(PROBE_PATH)
tab = probe.pivot_table(index=["model", "task"], columns="format", values="median_mass")[list(FORMATS)]
tab["chosen"] = [choose_format(probe[(probe.model == m) & (probe.task == t)]) for m, t in tab.index]
tab["v21_repro"] = probe[probe.format == "v21"].set_index(["model", "task"]).v21_repro_max_abs_dlog
tab.index = [f"{SHORT[m]} / {t}" for m, t in tab.index]
tab.to_csv(f"{OUT_DIR}/format_choice.csv")
tab.round(3)

### Before and after, for the 7 changed models

Calibrated clean accuracy, clean median label mass, tier (as in v2.2: *gated* = clean accuracy ≥ chance + 0.12; *reliable* = gated and label mass ≥ 0.5), and the normalised drop under `strip`.

In [ ]:
def brief(path, t):
    df = pd.read_csv(path)
    acc = (df.pred == df.label).groupby(df.cond).mean()
    ch = 1 / len(TASK_SPECS[t]["labels"])
    mass = df[df.cond == "clean"].label_mass.median()
    tier = "reliable" if acc["clean"] - ch >= 0.12 and mass >= 0.5 else ("gated" if acc["clean"] - ch >= 0.12 else "below gate")
    nd = (acc["clean"] - acc["strip"]) / (acc["clean"] - ch) if acc["clean"] - ch > 0.05 else np.nan
    return acc["clean"], mass, tier, nd

rows = []
for m in [m for m in MODELS if m in RESCORE or m in CHAT_TEMPLATE_FROM]:
    for t in TASKS:
        if not os.path.exists(pred_path(m, t)):
            continue
        a = brief(v21_path(m, t), t)
        b = brief(pred_path(m, t), t)
        rows.append({"model": SHORT[m], "task": t, "acc v2.1": a[0], "acc v3": b[0], "mass v2.1": a[1], "mass v3": b[1],
                     "tier v2.1": a[2], "tier v3": b[2], "nd_strip v2.1": a[3], "nd_strip v3": b[3]})
change = pd.DataFrame(rows)
change.to_csv(f"{OUT_DIR}/before_after.csv", index=False)
print("tiers, v2.1 → v3:", change.groupby(["tier v2.1", "tier v3"]).size().to_dict())
change.round(2)

### ViMMRC: NLL on the whole item vs on question + options

Per model: the perplexity ratio under `strip` (exp of the extra NLL per clean syllable) on both spans, and the item-level correlation of the two ΔNLL measures.

In [ ]:
rows = []
for m in MODELS:
    df = pd.read_csv(pred_path(m, "vimmrc")) if os.path.exists(pred_path(m, "vimmrc")) else None
    if df is None or "nll_qo" not in df:
        rows.append({"model": SHORT[m], "note": "no nll_qo"})
        continue
    g = df.set_index(["item_id", "cond"])
    full = (g.nll / g.clean_syl).unstack()
    qo = (g.nll_qo / g.qo_syl).unstack()
    rows.append({"model": SHORT[m], "ppl_ratio_full": np.exp((g.nll.unstack()["strip"].sum() - g.nll.unstack()["clean"].sum()) / g.clean_syl.unstack()["clean"].sum()),
                 "ppl_ratio_qo": np.exp((g.nll_qo.unstack()["strip"].sum() - g.nll_qo.unstack()["clean"].sum()) / g.qo_syl.unstack()["clean"].sum()),
                 "r_dnll_items": np.corrcoef(full["strip"] - full["clean"], qo["strip"] - qo["clean"])[0, 1]})
vim = pd.DataFrame(rows)
vim.to_csv(f"{OUT_DIR}/vimmrc_nll_spans.csv", index=False)
vim.round(3)

## 7. Next

1. Open **`1A_pilot_v3_analysis.ipynb`** on a **CPU** runtime and Run all. It is the v2.2 analysis pointed at `1A-pilot-v3/`, with ViMMRC's NLL taken from question + options. Outputs go to `MyDrive/llm-research/1A-pilot-v3-analysis/`.
2. `Runtime → Disconnect and delete runtime` here.